# Cleaning Dataset dan Pengujian Awal CBF + CARS TourHub

Notebook ini digunakan untuk membersihkan dataset utama:

`data/bali_tourist_destination.csv`

Output dataset bersih akan disimpan dengan nama:

`data/cleaned_dataaset_bali.csv`

Notebook ini dibuat untuk kebutuhan skripsi **rancang bangun aplikasi rekomendasi wisata Bali** dengan pendekatan:

- **Content-Based Filtering (CBF)**
- **Context-Aware Recommender System (CARS)**

Catatan penting:
- Notebook ini **bukan untuk prediksi cuaca**.
- Cuaca hanya digunakan sebagai **konteks** dalam CARS.
- Dataset yang dibersihkan hanya `bali_tourist_destination.csv`.


## 1. Import Library

In [1]:
import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

print("Library berhasil dimuat.")


Library berhasil dimuat.


## 2. Konfigurasi Path Dataset

Notebook ini dibuat agar fleksibel. Jika dijalankan dari folder root project atau dari folder `notebooks/`, file dataset tetap dicari otomatis.


In [2]:
DATASET_FILENAME = "bali_tourist_destination.csv"
OUTPUT_FILENAME = "cleaned_dataaset_bali.csv"
OUTPUT_SCENARIO_FILENAME = "hasil_test_skenario_cbf_cars.csv"

candidate_paths = [
    Path("data") / DATASET_FILENAME,
    Path("../data") / DATASET_FILENAME,
    Path(DATASET_FILENAME),
]

dataset_path = None
for path in candidate_paths:
    if path.exists():
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Pastikan file bali_tourist_destination.csv ada di folder data/ atau ../data/."
    )

data_dir = dataset_path.parent
output_path = data_dir / OUTPUT_FILENAME
scenario_output_path = data_dir / OUTPUT_SCENARIO_FILENAME

print("Dataset input :", dataset_path)
print("Dataset output:", output_path)
print("Output skenario:", scenario_output_path)


Dataset input : ../data/bali_tourist_destination.csv
Dataset output: ../data/cleaned_dataaset_bali.csv
Output skenario: ../data/hasil_test_skenario_cbf_cars.csv


## 3. Load Dataset Awal

In [3]:
df_raw = pd.read_csv(dataset_path)

print("Jumlah baris dan kolom dataset awal:", df_raw.shape)
display(df_raw.head())


Jumlah baris dan kolom dataset awal: (1452, 18)


,id_tempat,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,rating,jumlah_rating,latitude,longitude,link_google_maps,link_gambar,pexels_photo_id,pexels_photo_url,pexels_photographer,pexels_photographer_url,pexels_alt,pexels_query,pexels_source
0,BL0101001,Patung Titi Banda,Umum,Denpasar Barat,Kota Denpasar,4.6,1941,-8.649292,115.255043,https://www.google.com/maps/place/Patung+Titi+Banda/data=!4m7!3m6!1s0x2dd24075cfb1c9ab:0x504b91c49fa2dafa!8m2!3d-8.6...,https://images.pexels.com/photos/38189322/pexels-photo-38189322.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,38189322,https://www.pexels.com/id-id/foto/pengrajin-indonesia-mengukir-patung-kayu-38189322/,CoNa Konstan,https://www.pexels.com/id-id/@cona-konstan-528825662,Seorang perajin Indonesia dengan terampil mengukir patung kayu yang detail menggunakan tangan di Bali.,patung titi banda Bali,pexels_api
1,BL0101002,Uma.palak.(parkir.2),Rekreasi,Denpasar Barat,Kota Denpasar,4.8,83,-8.617877,115.212975,https://www.google.com/maps/place/Uma.palak.%28parkir.2%29/data=!4m7!3m6!1s0x2dd23f93096927d1:0xec812414518de557!8m2...,https://images.pexels.com/photos/34905643/pexels-photo-34905643.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,34905643,https://www.pexels.com/id-id/foto/34905643/,Ricky S,https://www.pexels.com/id-id/@ricky-s-2157293893,"Pasangan berpakaian tradisional Bali saat acara seremonial di Nusa Tenggara Barat, Indonesia.",uma palak Bali,pexels_api
2,BL0101003,Tukad Bindu Park,Rekreasi,Denpasar Barat,Kota Denpasar,4.5,1180,-8.643791,115.235812,https://www.google.com/maps/place/Tukad+Bindu+Park/data=!4m7!3m6!1s0x2dd23f789bdb40bb:0xc8f454047cb4a78!8m2!3d-8.643...,https://images.pexels.com/photos/35833514/pexels-photo-35833514.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,35833514,https://www.pexels.com/id-id/foto/upacara-adat-bali-di-pura-indonesia-35833514/,Hera Permata S,https://www.pexels.com/id-id/@hera-permata-s-2157462344,Upacara tradisional Bali bersama warga setempat di atas perahu dekat sebuah pura di Indonesia.,tukad bindu park Bali,pexels_api
3,BL0101004,Pantai Padang Galak,Alam,Denpasar Barat,Kota Denpasar,4.6,486,-8.661138,115.263301,https://www.google.com/maps/place/Pantai+Padang+Galak/data=!4m7!3m6!1s0x2dd241f0c7d5da97:0x1a904422c0294eea!8m2!3d-8...,https://images.pexels.com/photos/36299199/pexels-photo-36299199.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,36299199,https://www.pexels.com/id-id/foto/pemandangan-udara-indah-pantai-bali-dengan-mercusuar-36299199/,Tom Fisk,https://www.pexels.com/id-id/@tomfisk,Pemandangan udara pantai Bali dengan mercusuar dan garis pantai yang indah di Indonesia.,pantai padang galak Bali,pexels_api
4,BL0101005,Museum Le Mayeur,Budaya,Denpasar Barat,Kota Denpasar,4.1,679,-8.674877,115.263678,https://www.google.com/maps/place/Museum+Le+Mayeur/data=!4m7!3m6!1s0x2dd24039af93edab:0xbc2308b38e03f5c2!8m2!3d-8.67...,https://images.pexels.com/photos/35643903/pexels-photo-35643903.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,35643903,https://www.pexels.com/id-id/foto/patung-tradisional-bali-dengan-kipas-di-bandung-35643903/,picko pictura,https://www.pexels.com/id-id/@picko-pictura-2148902026,"Patung batu yang rumit depicting seorang penari Bali dengan kipas, berlatar belakang alam di Bandung, Indonesia.",museum le mayeur Bali,pexels_api


## 4. Pemeriksaan Struktur Dataset

Bagian ini digunakan untuk melihat struktur awal dataset, tipe data, jumlah kolom, dan informasi umum lainnya.


In [4]:
print("Daftar kolom dataset:")
for i, col in enumerate(df_raw.columns, start=1):
    print(f"{i}. {col}")

print("\nInformasi dataset:")
df_raw.info()


Daftar kolom dataset:
1. id_tempat
2. nama_tempat_wisata
3. kategori
4. kecamatan
5. kabupaten_kota
6. rating
7. jumlah_rating
8. latitude
9. longitude
10. link_google_maps
11. link_gambar
12. pexels_photo_id
13. pexels_photo_url
14. pexels_photographer
15. pexels_photographer_url
16. pexels_alt
17. pexels_query
18. pexels_source

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1452 entries, 0 to 1451
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id_tempat                1452 non-null   object 
 1   nama_tempat_wisata       1452 non-null   object 
 2   kategori                 1452 non-null   object 
 3   kecamatan                1452 non-null   object 
 4   kabupaten_kota           1452 non-null   object 
 5   rating                   1452 non-null   float64
 6   jumlah_rating            1452 non-null   int64  
 7   latitude                 1452 non-null   float64
 

## 5. Normalisasi Nama Kolom

Nama kolom dibuat seragam agar lebih aman diproses. Contohnya spasi diganti underscore dan huruf dibuat lowercase.


In [5]:
def normalize_column_name(column_name: str) -> str:
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name).strip("_")
    return column_name

df = df_raw.copy()
df.columns = [normalize_column_name(col) for col in df.columns]

print("Kolom setelah normalisasi:")
print(df.columns.tolist())
display(df.head())


Kolom setelah normalisasi:
['id_tempat', 'nama_tempat_wisata', 'kategori', 'kecamatan', 'kabupaten_kota', 'rating', 'jumlah_rating', 'latitude', 'longitude', 'link_google_maps', 'link_gambar', 'pexels_photo_id', 'pexels_photo_url', 'pexels_photographer', 'pexels_photographer_url', 'pexels_alt', 'pexels_query', 'pexels_source']


,id_tempat,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,rating,jumlah_rating,latitude,longitude,link_google_maps,link_gambar,pexels_photo_id,pexels_photo_url,pexels_photographer,pexels_photographer_url,pexels_alt,pexels_query,pexels_source
0,BL0101001,Patung Titi Banda,Umum,Denpasar Barat,Kota Denpasar,4.6,1941,-8.649292,115.255043,https://www.google.com/maps/place/Patung+Titi+Banda/data=!4m7!3m6!1s0x2dd24075cfb1c9ab:0x504b91c49fa2dafa!8m2!3d-8.6...,https://images.pexels.com/photos/38189322/pexels-photo-38189322.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,38189322,https://www.pexels.com/id-id/foto/pengrajin-indonesia-mengukir-patung-kayu-38189322/,CoNa Konstan,https://www.pexels.com/id-id/@cona-konstan-528825662,Seorang perajin Indonesia dengan terampil mengukir patung kayu yang detail menggunakan tangan di Bali.,patung titi banda Bali,pexels_api
1,BL0101002,Uma.palak.(parkir.2),Rekreasi,Denpasar Barat,Kota Denpasar,4.8,83,-8.617877,115.212975,https://www.google.com/maps/place/Uma.palak.%28parkir.2%29/data=!4m7!3m6!1s0x2dd23f93096927d1:0xec812414518de557!8m2...,https://images.pexels.com/photos/34905643/pexels-photo-34905643.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,34905643,https://www.pexels.com/id-id/foto/34905643/,Ricky S,https://www.pexels.com/id-id/@ricky-s-2157293893,"Pasangan berpakaian tradisional Bali saat acara seremonial di Nusa Tenggara Barat, Indonesia.",uma palak Bali,pexels_api
2,BL0101003,Tukad Bindu Park,Rekreasi,Denpasar Barat,Kota Denpasar,4.5,1180,-8.643791,115.235812,https://www.google.com/maps/place/Tukad+Bindu+Park/data=!4m7!3m6!1s0x2dd23f789bdb40bb:0xc8f454047cb4a78!8m2!3d-8.643...,https://images.pexels.com/photos/35833514/pexels-photo-35833514.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,35833514,https://www.pexels.com/id-id/foto/upacara-adat-bali-di-pura-indonesia-35833514/,Hera Permata S,https://www.pexels.com/id-id/@hera-permata-s-2157462344,Upacara tradisional Bali bersama warga setempat di atas perahu dekat sebuah pura di Indonesia.,tukad bindu park Bali,pexels_api
3,BL0101004,Pantai Padang Galak,Alam,Denpasar Barat,Kota Denpasar,4.6,486,-8.661138,115.263301,https://www.google.com/maps/place/Pantai+Padang+Galak/data=!4m7!3m6!1s0x2dd241f0c7d5da97:0x1a904422c0294eea!8m2!3d-8...,https://images.pexels.com/photos/36299199/pexels-photo-36299199.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,36299199,https://www.pexels.com/id-id/foto/pemandangan-udara-indah-pantai-bali-dengan-mercusuar-36299199/,Tom Fisk,https://www.pexels.com/id-id/@tomfisk,Pemandangan udara pantai Bali dengan mercusuar dan garis pantai yang indah di Indonesia.,pantai padang galak Bali,pexels_api
4,BL0101005,Museum Le Mayeur,Budaya,Denpasar Barat,Kota Denpasar,4.1,679,-8.674877,115.263678,https://www.google.com/maps/place/Museum+Le+Mayeur/data=!4m7!3m6!1s0x2dd24039af93edab:0xbc2308b38e03f5c2!8m2!3d-8.67...,https://images.pexels.com/photos/35643903/pexels-photo-35643903.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,35643903,https://www.pexels.com/id-id/foto/patung-tradisional-bali-dengan-kipas-di-bandung-35643903/,picko pictura,https://www.pexels.com/id-id/@picko-pictura-2148902026,"Patung batu yang rumit depicting seorang penari Bali dengan kipas, berlatar belakang alam di Bandung, Indonesia.",museum le mayeur Bali,pexels_api


## 6. Pemeriksaan Missing Value Sebelum Cleaning

In [6]:
missing_before = pd.DataFrame({
    "kolom": df.columns,
    "jumlah_missing": df.isna().sum().values,
    "persentase_missing": (df.isna().mean().values * 100).round(2)
}).sort_values("jumlah_missing", ascending=False)

display(missing_before)
print("Total missing value sebelum cleaning:", int(df.isna().sum().sum()))


,kolom,jumlah_missing,persentase_missing
0,id_tempat,0,0.0
1,nama_tempat_wisata,0,0.0
16,pexels_query,0,0.0
15,pexels_alt,0,0.0
14,pexels_photographer_url,0,0.0
13,pexels_photographer,0,0.0
12,pexels_photo_url,0,0.0
11,pexels_photo_id,0,0.0
10,link_gambar,0,0.0
9,link_google_maps,0,0.0


Total missing value sebelum cleaning: 0


## 7. Pemeriksaan Duplikat Sebelum Cleaning

In [7]:
duplicate_all_before = df.duplicated().sum()
print("Jumlah duplikat seluruh baris:", int(duplicate_all_before))

duplicate_subset_cols = [col for col in ["nama_tempat_wisata", "latitude", "longitude"] if col in df.columns]

if duplicate_subset_cols:
    duplicate_subset_before = df.duplicated(subset=duplicate_subset_cols).sum()
    print(f"Jumlah duplikat berdasarkan {duplicate_subset_cols}:", int(duplicate_subset_before))
else:
    print("Kolom untuk pengecekan duplikat nama/lokasi tidak lengkap.")


Jumlah duplikat seluruh baris: 0
Jumlah duplikat berdasarkan ['nama_tempat_wisata', 'latitude', 'longitude']: 5


## 8. Fungsi Cleaning Teks dan Angka

In [8]:
def clean_text_basic(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Samakan beberapa bentuk nilai kosong yang sering muncul di CSV.
    if value.lower() in ["", "-", "--", "nan", "none", "null", "n/a", "na"]:
        return np.nan

    # Rapikan spasi berlebih.
    value = re.sub(r"\s+", " ", value).strip()

    return value


def normalize_for_feature(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


def to_numeric_clean(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = value.replace(",", ".")
    value = re.sub(r"[^0-9.\-]", "", value)

    if value in ["", "-", ".", "-."]:
        return np.nan

    try:
        return float(value)
    except ValueError:
        return np.nan


print("Fungsi cleaning berhasil dibuat.")


Fungsi cleaning berhasil dibuat.


## 9. Cleaning Nilai Kosong dan Spasi Berlebih

Seluruh kolom bertipe teks dibersihkan dari spasi berlebih dan nilai kosong semu seperti `-`, `null`, `none`, dan sejenisnya.


In [9]:
clean = df.copy()

for col in clean.columns:
    if clean[col].dtype == "object":
        clean[col] = clean[col].apply(clean_text_basic)

display(clean.head())


,id_tempat,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,rating,jumlah_rating,latitude,longitude,link_google_maps,link_gambar,pexels_photo_id,pexels_photo_url,pexels_photographer,pexels_photographer_url,pexels_alt,pexels_query,pexels_source
0,BL0101001,Patung Titi Banda,Umum,Denpasar Barat,Kota Denpasar,4.6,1941,-8.649292,115.255043,https://www.google.com/maps/place/Patung+Titi+Banda/data=!4m7!3m6!1s0x2dd24075cfb1c9ab:0x504b91c49fa2dafa!8m2!3d-8.6...,https://images.pexels.com/photos/38189322/pexels-photo-38189322.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,38189322,https://www.pexels.com/id-id/foto/pengrajin-indonesia-mengukir-patung-kayu-38189322/,CoNa Konstan,https://www.pexels.com/id-id/@cona-konstan-528825662,Seorang perajin Indonesia dengan terampil mengukir patung kayu yang detail menggunakan tangan di Bali.,patung titi banda Bali,pexels_api
1,BL0101002,Uma.palak.(parkir.2),Rekreasi,Denpasar Barat,Kota Denpasar,4.8,83,-8.617877,115.212975,https://www.google.com/maps/place/Uma.palak.%28parkir.2%29/data=!4m7!3m6!1s0x2dd23f93096927d1:0xec812414518de557!8m2...,https://images.pexels.com/photos/34905643/pexels-photo-34905643.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,34905643,https://www.pexels.com/id-id/foto/34905643/,Ricky S,https://www.pexels.com/id-id/@ricky-s-2157293893,"Pasangan berpakaian tradisional Bali saat acara seremonial di Nusa Tenggara Barat, Indonesia.",uma palak Bali,pexels_api
2,BL0101003,Tukad Bindu Park,Rekreasi,Denpasar Barat,Kota Denpasar,4.5,1180,-8.643791,115.235812,https://www.google.com/maps/place/Tukad+Bindu+Park/data=!4m7!3m6!1s0x2dd23f789bdb40bb:0xc8f454047cb4a78!8m2!3d-8.643...,https://images.pexels.com/photos/35833514/pexels-photo-35833514.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,35833514,https://www.pexels.com/id-id/foto/upacara-adat-bali-di-pura-indonesia-35833514/,Hera Permata S,https://www.pexels.com/id-id/@hera-permata-s-2157462344,Upacara tradisional Bali bersama warga setempat di atas perahu dekat sebuah pura di Indonesia.,tukad bindu park Bali,pexels_api
3,BL0101004,Pantai Padang Galak,Alam,Denpasar Barat,Kota Denpasar,4.6,486,-8.661138,115.263301,https://www.google.com/maps/place/Pantai+Padang+Galak/data=!4m7!3m6!1s0x2dd241f0c7d5da97:0x1a904422c0294eea!8m2!3d-8...,https://images.pexels.com/photos/36299199/pexels-photo-36299199.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,36299199,https://www.pexels.com/id-id/foto/pemandangan-udara-indah-pantai-bali-dengan-mercusuar-36299199/,Tom Fisk,https://www.pexels.com/id-id/@tomfisk,Pemandangan udara pantai Bali dengan mercusuar dan garis pantai yang indah di Indonesia.,pantai padang galak Bali,pexels_api
4,BL0101005,Museum Le Mayeur,Budaya,Denpasar Barat,Kota Denpasar,4.1,679,-8.674877,115.263678,https://www.google.com/maps/place/Museum+Le+Mayeur/data=!4m7!3m6!1s0x2dd24039af93edab:0xbc2308b38e03f5c2!8m2!3d-8.67...,https://images.pexels.com/photos/35643903/pexels-photo-35643903.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,35643903,https://www.pexels.com/id-id/foto/patung-tradisional-bali-dengan-kipas-di-bandung-35643903/,picko pictura,https://www.pexels.com/id-id/@picko-pictura-2148902026,"Patung batu yang rumit depicting seorang penari Bali dengan kipas, berlatar belakang alam di Bandung, Indonesia.",museum le mayeur Bali,pexels_api


## 10. Validasi dan Cleaning Kolom Penting

Kolom yang diprioritaskan untuk skripsi TourHub:

- `nama_tempat_wisata`
- `kategori`
- `kecamatan`
- `kabupaten_kota`
- `rating`
- `jumlah_rating`
- `latitude`
- `longitude`
- `link_google_maps`
- `link_gambar`


In [10]:
required_columns = [
    "nama_tempat_wisata",
    "kategori",
    "kecamatan",
    "kabupaten_kota",
    "rating",
    "jumlah_rating",
    "latitude",
    "longitude",
    "link_google_maps",
    "link_gambar",
]

available_required = [col for col in required_columns if col in clean.columns]
missing_required = [col for col in required_columns if col not in clean.columns]

print("Kolom penting tersedia:", available_required)
print("Kolom penting tidak ditemukan:", missing_required)


Kolom penting tersedia: ['nama_tempat_wisata', 'kategori', 'kecamatan', 'kabupaten_kota', 'rating', 'jumlah_rating', 'latitude', 'longitude', 'link_google_maps', 'link_gambar']
Kolom penting tidak ditemukan: []


In [11]:
# Konversi kolom angka.
for numeric_col in ["rating", "jumlah_rating", "latitude", "longitude"]:
    if numeric_col in clean.columns:
        clean[numeric_col] = clean[numeric_col].apply(to_numeric_clean)

# Rating harus berada di rentang 0 sampai 5.
if "rating" in clean.columns:
    invalid_rating = clean[(clean["rating"].notna()) & ((clean["rating"] < 0) | (clean["rating"] > 5))]
    print("Jumlah rating di luar rentang 0-5 sebelum clip:", len(invalid_rating))
    clean["rating"] = clean["rating"].clip(lower=0, upper=5)

# Jumlah rating tidak boleh negatif.
if "jumlah_rating" in clean.columns:
    negative_reviews = clean[(clean["jumlah_rating"].notna()) & (clean["jumlah_rating"] < 0)]
    print("Jumlah jumlah_rating negatif sebelum cleaning:", len(negative_reviews))
    clean["jumlah_rating"] = clean["jumlah_rating"].fillna(0).clip(lower=0).round().astype(int)

# Jika rating kosong, gunakan median agar data tetap bisa diproses.
if "rating" in clean.columns:
    rating_median = clean["rating"].median()
    clean["rating"] = clean["rating"].fillna(rating_median)
    print("Median rating yang digunakan untuk mengisi rating kosong:", rating_median)

display(clean[available_required].head())


Jumlah rating di luar rentang 0-5 sebelum clip: 0
Jumlah jumlah_rating negatif sebelum cleaning: 0
Median rating yang digunakan untuk mengisi rating kosong: 4.6


,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,rating,jumlah_rating,latitude,longitude,link_google_maps,link_gambar
0,Patung Titi Banda,Umum,Denpasar Barat,Kota Denpasar,4.6,1941,-8.649292,115.255043,https://www.google.com/maps/place/Patung+Titi+Banda/data=!4m7!3m6!1s0x2dd24075cfb1c9ab:0x504b91c49fa2dafa!8m2!3d-8.6...,https://images.pexels.com/photos/38189322/pexels-photo-38189322.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200
1,Uma.palak.(parkir.2),Rekreasi,Denpasar Barat,Kota Denpasar,4.8,83,-8.617877,115.212975,https://www.google.com/maps/place/Uma.palak.%28parkir.2%29/data=!4m7!3m6!1s0x2dd23f93096927d1:0xec812414518de557!8m2...,https://images.pexels.com/photos/34905643/pexels-photo-34905643.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200
2,Tukad Bindu Park,Rekreasi,Denpasar Barat,Kota Denpasar,4.5,1180,-8.643791,115.235812,https://www.google.com/maps/place/Tukad+Bindu+Park/data=!4m7!3m6!1s0x2dd23f789bdb40bb:0xc8f454047cb4a78!8m2!3d-8.643...,https://images.pexels.com/photos/35833514/pexels-photo-35833514.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200
3,Pantai Padang Galak,Alam,Denpasar Barat,Kota Denpasar,4.6,486,-8.661138,115.263301,https://www.google.com/maps/place/Pantai+Padang+Galak/data=!4m7!3m6!1s0x2dd241f0c7d5da97:0x1a904422c0294eea!8m2!3d-8...,https://images.pexels.com/photos/36299199/pexels-photo-36299199.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200
4,Museum Le Mayeur,Budaya,Denpasar Barat,Kota Denpasar,4.1,679,-8.674877,115.263678,https://www.google.com/maps/place/Museum+Le+Mayeur/data=!4m7!3m6!1s0x2dd24039af93edab:0xbc2308b38e03f5c2!8m2!3d-8.67...,https://images.pexels.com/photos/35643903/pexels-photo-35643903.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200


## 11. Validasi Koordinat Area Bali

Validasi koordinat dilakukan untuk memastikan destinasi berada pada rentang wilayah Bali secara umum.

Rentang yang digunakan dibuat cukup longgar:

- Latitude: `-9.2` sampai `-7.5`
- Longitude: `114.0` sampai `116.0`

Jika ada data di luar rentang ini, data tersebut ditandai dan dapat dihapus dari dataset bersih.


In [12]:
BALI_LAT_MIN = -9.2
BALI_LAT_MAX = -7.5
BALI_LON_MIN = 114.0
BALI_LON_MAX = 116.0

if {"latitude", "longitude"}.issubset(clean.columns):
    clean["is_valid_bali_coordinate"] = (
        clean["latitude"].between(BALI_LAT_MIN, BALI_LAT_MAX)
        & clean["longitude"].between(BALI_LON_MIN, BALI_LON_MAX)
    )

    invalid_coords = clean[~clean["is_valid_bali_coordinate"]]
    print("Jumlah data dengan koordinat di luar area Bali:", len(invalid_coords))

    if len(invalid_coords) > 0:
        display_cols = [col for col in ["nama_tempat_wisata", "kabupaten_kota", "latitude", "longitude"] if col in clean.columns]
        display(invalid_coords[display_cols].head(20))
else:
    print("Kolom latitude/longitude tidak lengkap, validasi koordinat dilewati.")


Jumlah data dengan koordinat di luar area Bali: 7


,nama_tempat_wisata,kabupaten_kota,latitude,longitude
661,Goa Selomangleng,Kabupaten Tabanan,-7.807188,111.972672
670,Taman Wisata Tirtoyoso Park,Kabupaten Tabanan,-7.816438,112.029748
673,Air Terjun Dolo,Kabupaten Tabanan,-7.867689,111.833469
678,Gumul Paradise Island,Kabupaten Tabanan,-7.811681,112.061169
682,Taman Hutan Joyoboyo Kediri,Kabupaten Tabanan,-7.818138,112.029716
684,Wisata Sempu Exotic Park,Kabupaten Tabanan,-7.960526,112.217889
688,Air Terjun Irenggolo,Kabupaten Tabanan,-7.863541,111.850352


## 12. Penghapusan Data Tidak Valid dan Duplikat

Tahap ini membersihkan:

1. Data tanpa nama destinasi.
2. Data dengan koordinat tidak valid.
3. Data duplikat berdasarkan nama destinasi dan koordinat.


In [13]:
rows_before_cleaning = len(clean)

# Hapus data tanpa nama destinasi.
if "nama_tempat_wisata" in clean.columns:
    clean = clean.dropna(subset=["nama_tempat_wisata"])

# Hapus data dengan koordinat tidak valid.
if "is_valid_bali_coordinate" in clean.columns:
    clean = clean[clean["is_valid_bali_coordinate"] == True].copy()

# Hapus data duplikat.
if {"nama_tempat_wisata", "latitude", "longitude"}.issubset(clean.columns):
    clean = clean.drop_duplicates(subset=["nama_tempat_wisata", "latitude", "longitude"], keep="first")
else:
    clean = clean.drop_duplicates(keep="first")

rows_after_cleaning = len(clean)

print("Jumlah data sebelum cleaning utama:", rows_before_cleaning)
print("Jumlah data setelah cleaning utama:", rows_after_cleaning)
print("Jumlah data yang dihapus:", rows_before_cleaning - rows_after_cleaning)


Jumlah data sebelum cleaning utama: 1452
Jumlah data setelah cleaning utama: 1440
Jumlah data yang dihapus: 12


## 13. Pembuatan Kolom `tipe_wisata`

Kolom `tipe_wisata` digunakan untuk kebutuhan CARS.

Nilainya:

- `outdoor`
- `indoor`
- `mixed`

Jika dataset sudah punya kolom `tipe_wisata`, nilainya akan dirapikan. Jika belum ada, sistem akan melakukan inferensi sederhana berdasarkan nama/kategori destinasi.


In [14]:
OUTDOOR_KEYWORDS = [
    "pantai", "beach", "waterfall", "air terjun", "gunung", "mount", "bukit", "hill",
    "danau", "lake", "sawah", "rice", "campuhan", "camping", "park", "taman",
    "rafting", "snorkeling", "diving", "zoo", "safari", "desa wisata", "hutan",
    "forest", "tebing", "cliff", "pura", "temple", "tirta", "goa", "cave"
]

INDOOR_KEYWORDS = [
    "museum", "gallery", "galeri", "mall", "pasar", "market", "studio", "spa",
    "restaurant", "restoran", "cafe", "coffee", "hotel", "resort", "workshop",
    "pusat oleh", "art center", "rumah makan"
]

MIXED_KEYWORDS = [
    "cultural", "budaya", "village", "desa", "agrowisata", "edukasi", "tour",
    "waterpark", "water park", "wisata keluarga"
]


def infer_tourism_type(row):
    if "tipe_wisata" in row.index and pd.notna(row.get("tipe_wisata")):
        existing = str(row.get("tipe_wisata")).strip().lower()
        if existing in ["outdoor", "luar ruangan"]:
            return "outdoor"
        if existing in ["indoor", "dalam ruangan"]:
            return "indoor"
        if existing in ["mixed", "campuran"]:
            return "mixed"

    text_parts = []
    for col in ["nama_tempat_wisata", "kategori", "kecamatan", "kabupaten_kota"]:
        if col in row.index and pd.notna(row.get(col)):
            text_parts.append(str(row.get(col)).lower())

    text = " ".join(text_parts)

    indoor_score = sum(1 for kw in INDOOR_KEYWORDS if kw in text)
    outdoor_score = sum(1 for kw in OUTDOOR_KEYWORDS if kw in text)
    mixed_score = sum(1 for kw in MIXED_KEYWORDS if kw in text)

    if mixed_score > 0 and outdoor_score > 0:
        return "mixed"
    if indoor_score > outdoor_score:
        return "indoor"
    if outdoor_score > indoor_score:
        return "outdoor"
    if mixed_score > 0:
        return "mixed"

    # Default wisata di Bali banyak yang outdoor, tetapi dibuat mixed agar lebih netral.
    return "mixed"


clean["tipe_wisata"] = clean.apply(infer_tourism_type, axis=1)

print("Distribusi tipe_wisata:")
display(clean["tipe_wisata"].value_counts().reset_index().rename(columns={"index": "tipe_wisata", "tipe_wisata": "jumlah"}))


Distribusi tipe_wisata:


,jumlah,count
0,mixed,729
1,outdoor,614
2,indoor,97


## 14. Pembuatan Skor Tambahan untuk Rekomendasi

Skor tambahan yang dibuat:

- `rating_score`: normalisasi rating ke rentang 0 sampai 1.
- `popularity_score`: normalisasi jumlah ulasan menggunakan log agar tidak terlalu berat ke destinasi yang sangat populer.


In [15]:
if "rating" in clean.columns:
    clean["rating_score"] = (clean["rating"] / 5).clip(0, 1)
else:
    clean["rating_score"] = 0.0

if "jumlah_rating" in clean.columns:
    max_log_review = np.log1p(clean["jumlah_rating"].max())
    if max_log_review == 0:
        clean["popularity_score"] = 0.0
    else:
        clean["popularity_score"] = (np.log1p(clean["jumlah_rating"]) / max_log_review).clip(0, 1)
else:
    clean["popularity_score"] = 0.0

display(clean[[col for col in ["nama_tempat_wisata", "rating", "jumlah_rating", "rating_score", "popularity_score"] if col in clean.columns]].head())


,nama_tempat_wisata,rating,jumlah_rating,rating_score,popularity_score
0,Patung Titi Banda,4.6,1941,0.92,0.656787
1,Uma.palak.(parkir.2),4.8,83,0.96,0.384351
2,Tukad Bindu Park,4.5,1180,0.90,0.613644
3,Pantai Padang Galak,4.6,486,0.92,0.536801
4,Museum Le Mayeur,4.1,679,0.82,0.565759


## 15. Pembuatan `feature_text` untuk Content-Based Filtering

Kolom `feature_text` menjadi representasi teks destinasi wisata untuk proses CBF.

Kolom yang digabungkan:

- nama tempat wisata
- kategori
- kecamatan
- kabupaten/kota
- tipe wisata


In [16]:
feature_columns = [col for col in ["nama_tempat_wisata", "kategori", "kecamatan", "kabupaten_kota", "tipe_wisata"] if col in clean.columns]

def build_feature_text(row):
    text_parts = []
    for col in feature_columns:
        text_parts.append(normalize_for_feature(row.get(col, "")))
    return " ".join([part for part in text_parts if part]).strip()

clean["feature_text"] = clean.apply(build_feature_text, axis=1)

display(clean[[col for col in ["nama_tempat_wisata", "kategori", "kecamatan", "kabupaten_kota", "tipe_wisata", "feature_text"] if col in clean.columns]].head())


,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,tipe_wisata,feature_text
0,Patung Titi Banda,Umum,Denpasar Barat,Kota Denpasar,indoor,patung titi banda umum denpasar barat kota denpasar indoor
1,Uma.palak.(parkir.2),Rekreasi,Denpasar Barat,Kota Denpasar,mixed,uma palak parkir 2 rekreasi denpasar barat kota denpasar mixed
2,Tukad Bindu Park,Rekreasi,Denpasar Barat,Kota Denpasar,mixed,tukad bindu park rekreasi denpasar barat kota denpasar mixed
3,Pantai Padang Galak,Alam,Denpasar Barat,Kota Denpasar,mixed,pantai padang galak alam denpasar barat kota denpasar mixed
4,Museum Le Mayeur,Budaya,Denpasar Barat,Kota Denpasar,indoor,museum le mayeur budaya denpasar barat kota denpasar indoor


## 16. Pemeriksaan Missing Value Setelah Cleaning

In [17]:
missing_after = pd.DataFrame({
    "kolom": clean.columns,
    "jumlah_missing": clean.isna().sum().values,
    "persentase_missing": (clean.isna().mean().values * 100).round(2)
}).sort_values("jumlah_missing", ascending=False)

display(missing_after)
print("Total missing value setelah cleaning:", int(clean.isna().sum().sum()))


,kolom,jumlah_missing,persentase_missing
0,id_tempat,0,0.0
12,pexels_photo_url,0,0.0
21,popularity_score,0,0.0
20,rating_score,0,0.0
19,tipe_wisata,0,0.0
18,is_valid_bali_coordinate,0,0.0
17,pexels_source,0,0.0
16,pexels_query,0,0.0
15,pexels_alt,0,0.0
14,pexels_photographer_url,0,0.0


Total missing value setelah cleaning: 0


## 17. Ringkasan Hasil Cleaning Dataset

In [18]:
summary_cleaning = pd.DataFrame([
    {"Keterangan": "Jumlah data awal", "Nilai": len(df)},
    {"Keterangan": "Jumlah data setelah cleaning", "Nilai": len(clean)},
    {"Keterangan": "Jumlah data dihapus", "Nilai": len(df) - len(clean)},
    {"Keterangan": "Jumlah kolom awal", "Nilai": df.shape[1]},
    {"Keterangan": "Jumlah kolom setelah cleaning", "Nilai": clean.shape[1]},
    {"Keterangan": "Total missing value awal", "Nilai": int(df.isna().sum().sum())},
    {"Keterangan": "Total missing value setelah cleaning", "Nilai": int(clean.isna().sum().sum())},
])

display(summary_cleaning)


,Keterangan,Nilai
0,Jumlah data awal,1452
1,Jumlah data setelah cleaning,1440
2,Jumlah data dihapus,12
3,Jumlah kolom awal,18
4,Jumlah kolom setelah cleaning,23
5,Total missing value awal,0
6,Total missing value setelah cleaning,0


## 18. Simpan Dataset Bersih

Dataset bersih disimpan dengan nama:

`cleaned_dataaset_bali.csv`

File ini yang dapat digunakan untuk aplikasi atau dokumentasi skripsi.


In [19]:
# Hapus kolom bantu validasi jika tidak ingin ikut ke dataset final.
final_clean = clean.copy()

# Kolom is_valid_bali_coordinate boleh tetap disimpan sebagai bukti validasi.
# Jika ingin dihapus, aktifkan baris di bawah:
# final_clean = final_clean.drop(columns=["is_valid_bali_coordinate"], errors="ignore")

final_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Dataset bersih berhasil disimpan ke:")
print(output_path)

display(final_clean.head())


Dataset bersih berhasil disimpan ke:
../data/cleaned_dataaset_bali.csv


,id_tempat,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,rating,jumlah_rating,latitude,longitude,link_google_maps,link_gambar,pexels_photo_id,pexels_photo_url,pexels_photographer,pexels_photographer_url,pexels_alt,pexels_query,pexels_source,is_valid_bali_coordinate,tipe_wisata,rating_score,popularity_score,feature_text
0,BL0101001,Patung Titi Banda,Umum,Denpasar Barat,Kota Denpasar,4.6,1941,-8.649292,115.255043,https://www.google.com/maps/place/Patung+Titi+Banda/data=!4m7!3m6!1s0x2dd24075cfb1c9ab:0x504b91c49fa2dafa!8m2!3d-8.6...,https://images.pexels.com/photos/38189322/pexels-photo-38189322.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,38189322,https://www.pexels.com/id-id/foto/pengrajin-indonesia-mengukir-patung-kayu-38189322/,CoNa Konstan,https://www.pexels.com/id-id/@cona-konstan-528825662,Seorang perajin Indonesia dengan terampil mengukir patung kayu yang detail menggunakan tangan di Bali.,patung titi banda Bali,pexels_api,True,indoor,0.92,0.656787,patung titi banda umum denpasar barat kota denpasar indoor
1,BL0101002,Uma.palak.(parkir.2),Rekreasi,Denpasar Barat,Kota Denpasar,4.8,83,-8.617877,115.212975,https://www.google.com/maps/place/Uma.palak.%28parkir.2%29/data=!4m7!3m6!1s0x2dd23f93096927d1:0xec812414518de557!8m2...,https://images.pexels.com/photos/34905643/pexels-photo-34905643.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,34905643,https://www.pexels.com/id-id/foto/34905643/,Ricky S,https://www.pexels.com/id-id/@ricky-s-2157293893,"Pasangan berpakaian tradisional Bali saat acara seremonial di Nusa Tenggara Barat, Indonesia.",uma palak Bali,pexels_api,True,mixed,0.96,0.384351,uma palak parkir 2 rekreasi denpasar barat kota denpasar mixed
2,BL0101003,Tukad Bindu Park,Rekreasi,Denpasar Barat,Kota Denpasar,4.5,1180,-8.643791,115.235812,https://www.google.com/maps/place/Tukad+Bindu+Park/data=!4m7!3m6!1s0x2dd23f789bdb40bb:0xc8f454047cb4a78!8m2!3d-8.643...,https://images.pexels.com/photos/35833514/pexels-photo-35833514.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,35833514,https://www.pexels.com/id-id/foto/upacara-adat-bali-di-pura-indonesia-35833514/,Hera Permata S,https://www.pexels.com/id-id/@hera-permata-s-2157462344,Upacara tradisional Bali bersama warga setempat di atas perahu dekat sebuah pura di Indonesia.,tukad bindu park Bali,pexels_api,True,mixed,0.90,0.613644,tukad bindu park rekreasi denpasar barat kota denpasar mixed
3,BL0101004,Pantai Padang Galak,Alam,Denpasar Barat,Kota Denpasar,4.6,486,-8.661138,115.263301,https://www.google.com/maps/place/Pantai+Padang+Galak/data=!4m7!3m6!1s0x2dd241f0c7d5da97:0x1a904422c0294eea!8m2!3d-8...,https://images.pexels.com/photos/36299199/pexels-photo-36299199.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,36299199,https://www.pexels.com/id-id/foto/pemandangan-udara-indah-pantai-bali-dengan-mercusuar-36299199/,Tom Fisk,https://www.pexels.com/id-id/@tomfisk,Pemandangan udara pantai Bali dengan mercusuar dan garis pantai yang indah di Indonesia.,pantai padang galak Bali,pexels_api,True,mixed,0.92,0.536801,pantai padang galak alam denpasar barat kota denpasar mixed
4,BL0101005,Museum Le Mayeur,Budaya,Denpasar Barat,Kota Denpasar,4.1,679,-8.674877,115.263678,https://www.google.com/maps/place/Museum+Le+Mayeur/data=!4m7!3m6!1s0x2dd24039af93edab:0xbc2308b38e03f5c2!8m2!3d-8.67...,https://images.pexels.com/photos/35643903/pexels-photo-35643903.jpeg?auto=compress&cs=tinysrgb&fit=crop&h=627&w=1200,35643903,https://www.pexels.com/id-id/foto/patung-tradisional-bali-dengan-kipas-di-bandung-35643903/,picko pictura,https://www.pexels.com/id-id/@picko-pictura-2148902026,"Patung batu yang rumit depicting seorang penari Bali dengan kipas, berlatar belakang alam di Bandung, Indonesia.",museum le mayeur Bali,pexels_api,True,indoor,0.82,0.565759,museum le mayeur budaya denpasar barat kota denpasar indoor


# Pengujian Awal Dataset untuk CBF + CARS

Bagian ini digunakan sebagai bukti bahwa dataset yang sudah dibersihkan dapat digunakan untuk sistem rekomendasi.


## 19. Membuat Model TF-IDF untuk Content-Based Filtering

In [20]:
tfidf = TfidfVectorizer(
    stop_words=None,
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf.fit_transform(final_clean["feature_text"].fillna(""))

print("Ukuran matrix TF-IDF:", tfidf_matrix.shape)
print("Jumlah fitur TF-IDF:", len(tfidf.get_feature_names_out()))


Ukuran matrix TF-IDF: (1440, 5610)
Jumlah fitur TF-IDF: 5610


## 20. Fungsi Rekomendasi CBF

CBF digunakan untuk mencari destinasi yang paling mirip dengan preferensi pengguna.


In [21]:
def recommend_cbf(query, top_n=10):
    query_clean = normalize_for_feature(query)
    query_vector = tfidf.transform([query_clean])

    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    result = final_clean.copy()
    result["cbf_score"] = similarity_scores

    result["base_score"] = (
        0.70 * result["cbf_score"]
        + 0.20 * result["rating_score"]
        + 0.10 * result["popularity_score"]
    )

    result = result.sort_values("base_score", ascending=False)

    display_cols = [
        "nama_tempat_wisata",
        "kategori",
        "kecamatan",
        "kabupaten_kota",
        "tipe_wisata",
        "rating",
        "jumlah_rating",
        "cbf_score",
        "base_score",
    ]
    display_cols = [col for col in display_cols if col in result.columns]

    return result[display_cols].head(top_n)


test_cbf = recommend_cbf("wisata alam pantai sunset outdoor", top_n=10)
display(test_cbf)


,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,tipe_wisata,rating,jumlah_rating,cbf_score,base_score
1052,Wisata Alam Gredeg,Umum,Tembuku,Kabupaten Bangli,mixed,5.0,8,0.279912,0.414998
703,Wisata Alam Dudu,Umum,Kerambitan,Kabupaten Tabanan,mixed,4.8,8,0.289009,0.413366
708,Baturiti Bersemi Desa Wisata,Alam,Kerambitan,Kabupaten Tabanan,mixed,5.0,1,0.268732,0.394125
33,Obyek Wisata Alam Green Cliff,Umum,Denpasar Barat,Kota Denpasar,mixed,4.1,173,0.239718,0.376555
815,Taman Wisata Alam Danau Buyan - Danau Tamblingan,Alam,Buleleng,Kabupaten Buleleng,outdoor,4.4,60,0.234979,0.376145
270,Sunset Point Pantai Cemongkak,Alam,Kuta Selatan,Kabupaten Badung,outdoor,4.6,47,0.221526,0.372649
1179,Sunset Point,Umum,Nusa Penida,Kabupaten Klungkung,mixed,4.6,597,0.185468,0.369289
1028,Wisata Alam kutuh,Umum,Susut,Kabupaten Bangli,mixed,3.7,3,0.293279,0.365321
201,Pantai Pandawa,Alam,Kuta,Kabupaten Badung,outdoor,4.6,44808,0.112385,0.355575
250,Pantai Kuta,Alam,Kuta,Kabupaten Badung,outdoor,4.5,42383,0.115759,0.353454


## 21. Fungsi Context-Aware Recommender System

CARS digunakan untuk menyesuaikan skor rekomendasi berdasarkan konteks.

Konteks yang digunakan:

- cuaca
- weekend
- high season
- tipe wisata
- popularitas destinasi

Catatan:
CARS **tidak memprediksi cuaca**. CARS hanya memakai kondisi cuaca sebagai input konteks.


In [22]:
def normalize_weather(weather):
    weather = str(weather or "").lower().strip()

    if any(word in weather for word in ["hujan", "rain", "storm", "petir"]):
        return "hujan"
    if any(word in weather for word in ["cerah", "clear", "sunny"]):
        return "cerah"
    if any(word in weather for word in ["berawan", "cloud", "mendung"]):
        return "berawan"

    return "tidak_diketahui"


def context_multiplier(row, weather="cerah", is_weekend=False, is_high_season=False):
    weather = normalize_weather(weather)
    tourism_type = str(row.get("tipe_wisata", "mixed")).lower()
    popularity = float(row.get("popularity_score", 0))

    multiplier = 1.0
    reasons = []

    # Konteks cuaca
    if weather == "hujan":
        if tourism_type == "outdoor":
            multiplier *= 0.72
            reasons.append("cuaca hujan kurang cocok untuk destinasi luar ruangan")
        elif tourism_type == "indoor":
            multiplier *= 1.12
            reasons.append("cuaca hujan lebih cocok untuk destinasi dalam ruangan")
        else:
            multiplier *= 0.92
            reasons.append("cuaca hujan cukup aman untuk destinasi campuran")
    elif weather == "cerah":
        if tourism_type == "outdoor":
            multiplier *= 1.10
            reasons.append("cuaca cerah cocok untuk destinasi luar ruangan")
        elif tourism_type == "indoor":
            multiplier *= 0.96
            reasons.append("cuaca cerah tetap dapat digunakan untuk destinasi dalam ruangan")
        else:
            multiplier *= 1.04
            reasons.append("cuaca cerah cukup cocok untuk destinasi campuran")
    elif weather == "berawan":
        if tourism_type == "outdoor":
            multiplier *= 1.02
            reasons.append("cuaca berawan masih cukup cocok untuk destinasi luar ruangan")
        else:
            multiplier *= 1.00
            reasons.append("cuaca berawan tidak memberi perubahan besar")

    # Konteks weekend dan high season
    if is_weekend or is_high_season:
        if popularity >= 0.75:
            multiplier *= 0.88
            reasons.append("akhir pekan atau musim liburan membuat destinasi populer berpotensi lebih ramai")
        else:
            multiplier *= 1.04
            reasons.append("akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman")

    return multiplier, "; ".join(reasons)


print("Fungsi CARS berhasil dibuat.")


Fungsi CARS berhasil dibuat.


## 22. Fungsi Rekomendasi CBF + CARS

Hasil CBF disesuaikan menggunakan konteks CARS sehingga menghasilkan `final_score`.


In [23]:
def recommend_cbf_cars(query, weather="cerah", is_weekend=False, is_high_season=False, top_n=10):
    query_clean = normalize_for_feature(query)
    query_vector = tfidf.transform([query_clean])
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    result = final_clean.copy()
    result["cbf_score"] = similarity_scores

    result["base_score"] = (
        0.70 * result["cbf_score"]
        + 0.20 * result["rating_score"]
        + 0.10 * result["popularity_score"]
    )

    multipliers = result.apply(
        lambda row: context_multiplier(
            row,
            weather=weather,
            is_weekend=is_weekend,
            is_high_season=is_high_season
        ),
        axis=1
    )

    result["context_multiplier"] = [item[0] for item in multipliers]
    result["reason"] = [item[1] for item in multipliers]
    result["final_score"] = result["base_score"] * result["context_multiplier"]

    result = result.sort_values("final_score", ascending=False)

    display_cols = [
        "nama_tempat_wisata",
        "kategori",
        "kecamatan",
        "kabupaten_kota",
        "tipe_wisata",
        "rating",
        "jumlah_rating",
        "cbf_score",
        "base_score",
        "context_multiplier",
        "final_score",
        "reason",
    ]
    display_cols = [col for col in display_cols if col in result.columns]

    return result[display_cols].head(top_n)


test_cbf_cars = recommend_cbf_cars(
    query="wisata alam pantai sunset outdoor",
    weather="cerah",
    is_weekend=True,
    is_high_season=False,
    top_n=10
)

display(test_cbf_cars)


,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,tipe_wisata,rating,jumlah_rating,cbf_score,base_score,context_multiplier,final_score,reason
1052,Wisata Alam Gredeg,Umum,Tembuku,Kabupaten Bangli,mixed,5.0,8,0.279912,0.414998,1.0816,0.448862,cuaca cerah cukup cocok untuk destinasi campuran; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
703,Wisata Alam Dudu,Umum,Kerambitan,Kabupaten Tabanan,mixed,4.8,8,0.289009,0.413366,1.0816,0.447097,cuaca cerah cukup cocok untuk destinasi campuran; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
815,Taman Wisata Alam Danau Buyan - Danau Tamblingan,Alam,Buleleng,Kabupaten Buleleng,outdoor,4.4,60,0.234979,0.376145,1.1440,0.430310,cuaca cerah cocok untuk destinasi luar ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
270,Sunset Point Pantai Cemongkak,Alam,Kuta Selatan,Kabupaten Badung,outdoor,4.6,47,0.221526,0.372649,1.1440,0.426311,cuaca cerah cocok untuk destinasi luar ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
708,Baturiti Bersemi Desa Wisata,Alam,Kerambitan,Kabupaten Tabanan,mixed,5.0,1,0.268732,0.394125,1.0816,0.426286,cuaca cerah cukup cocok untuk destinasi campuran; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
33,Obyek Wisata Alam Green Cliff,Umum,Denpasar Barat,Kota Denpasar,mixed,4.1,173,0.239718,0.376555,1.0816,0.407282,cuaca cerah cukup cocok untuk destinasi campuran; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
1179,Sunset Point,Umum,Nusa Penida,Kabupaten Klungkung,mixed,4.6,597,0.185468,0.369289,1.0816,0.399423,cuaca cerah cukup cocok untuk destinasi campuran; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
1028,Wisata Alam kutuh,Umum,Susut,Kabupaten Bangli,mixed,3.7,3,0.293279,0.365321,1.0816,0.395131,cuaca cerah cukup cocok untuk destinasi campuran; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
810,Wisata Pantai penimbangan,Alam,Buleleng,Kabupaten Buleleng,outdoor,4.8,20,0.164044,0.333240,1.1440,0.381227,cuaca cerah cocok untuk destinasi luar ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
219,Pantai Mengening,Alam,Kuta,Kabupaten Badung,outdoor,4.7,2025,0.108896,0.330273,1.1440,0.377833,cuaca cerah cocok untuk destinasi luar ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman


## 23. Pengujian Beberapa Skenario Rekomendasi

Bagian ini dapat digunakan sebagai bukti pengujian awal algoritma pada skripsi.

Skenario yang diuji:

1. Wisata alam saat cuaca cerah.
2. Wisata budaya saat hujan.
3. Wisata keluarga saat weekend dan high season.
4. Wisata pantai saat hujan.


In [24]:
scenarios = [
    {
        "nama_skenario": "Skenario 1 - Wisata alam saat cuaca cerah",
        "query": "wisata alam pantai sunset outdoor",
        "weather": "cerah",
        "is_weekend": False,
        "is_high_season": False,
    },
    {
        "nama_skenario": "Skenario 2 - Wisata budaya saat hujan",
        "query": "wisata budaya museum pura sejarah indoor",
        "weather": "hujan",
        "is_weekend": False,
        "is_high_season": False,
    },
    {
        "nama_skenario": "Skenario 3 - Wisata keluarga saat weekend dan high season",
        "query": "wisata keluarga rekreasi taman edukasi",
        "weather": "berawan",
        "is_weekend": True,
        "is_high_season": True,
    },
    {
        "nama_skenario": "Skenario 4 - Wisata pantai saat hujan",
        "query": "wisata pantai laut outdoor",
        "weather": "hujan",
        "is_weekend": True,
        "is_high_season": False,
    },
]

all_scenario_results = []

for scenario in scenarios:
    print("\n" + "=" * 100)
    print(scenario["nama_skenario"])
    print("=" * 100)

    result = recommend_cbf_cars(
        query=scenario["query"],
        weather=scenario["weather"],
        is_weekend=scenario["is_weekend"],
        is_high_season=scenario["is_high_season"],
        top_n=5
    )

    result = result.copy()
    result.insert(0, "nama_skenario", scenario["nama_skenario"])
    result.insert(1, "query", scenario["query"])
    result.insert(2, "weather", scenario["weather"])
    result.insert(3, "is_weekend", scenario["is_weekend"])
    result.insert(4, "is_high_season", scenario["is_high_season"])

    display(result)
    all_scenario_results.append(result)

scenario_results_df = pd.concat(all_scenario_results, ignore_index=True)
scenario_results_df.to_csv(scenario_output_path, index=False, encoding="utf-8-sig")

print("\nHasil pengujian skenario berhasil disimpan ke:")
print(scenario_output_path)



Skenario 1 - Wisata alam saat cuaca cerah


,nama_skenario,query,weather,is_weekend,is_high_season,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,tipe_wisata,rating,jumlah_rating,cbf_score,base_score,context_multiplier,final_score,reason
1052,Skenario 1 - Wisata alam saat cuaca cerah,wisata alam pantai sunset outdoor,cerah,False,False,Wisata Alam Gredeg,Umum,Tembuku,Kabupaten Bangli,mixed,5.0,8,0.279912,0.414998,1.04,0.431598,cuaca cerah cukup cocok untuk destinasi campuran
703,Skenario 1 - Wisata alam saat cuaca cerah,wisata alam pantai sunset outdoor,cerah,False,False,Wisata Alam Dudu,Umum,Kerambitan,Kabupaten Tabanan,mixed,4.8,8,0.289009,0.413366,1.04,0.429901,cuaca cerah cukup cocok untuk destinasi campuran
815,Skenario 1 - Wisata alam saat cuaca cerah,wisata alam pantai sunset outdoor,cerah,False,False,Taman Wisata Alam Danau Buyan - Danau Tamblingan,Alam,Buleleng,Kabupaten Buleleng,outdoor,4.4,60,0.234979,0.376145,1.10,0.413759,cuaca cerah cocok untuk destinasi luar ruangan
270,Skenario 1 - Wisata alam saat cuaca cerah,wisata alam pantai sunset outdoor,cerah,False,False,Sunset Point Pantai Cemongkak,Alam,Kuta Selatan,Kabupaten Badung,outdoor,4.6,47,0.221526,0.372649,1.10,0.409914,cuaca cerah cocok untuk destinasi luar ruangan
708,Skenario 1 - Wisata alam saat cuaca cerah,wisata alam pantai sunset outdoor,cerah,False,False,Baturiti Bersemi Desa Wisata,Alam,Kerambitan,Kabupaten Tabanan,mixed,5.0,1,0.268732,0.394125,1.04,0.409890,cuaca cerah cukup cocok untuk destinasi campuran



Skenario 2 - Wisata budaya saat hujan


,nama_skenario,query,weather,is_weekend,is_high_season,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,tipe_wisata,rating,jumlah_rating,cbf_score,base_score,context_multiplier,final_score,reason
98,Skenario 2 - Wisata budaya saat hujan,wisata budaya museum pura sejarah indoor,hujan,False,False,Museum Bali,Budaya,Denpasar Timur,Kota Denpasar,indoor,4.3,3223,0.326072,0.470326,1.12,0.526766,cuaca hujan lebih cocok untuk destinasi dalam ruangan
118,Skenario 2 - Wisata budaya saat hujan,wisata budaya museum pura sejarah indoor,hujan,False,False,Museum Bung Karno,Budaya,Denpasar Timur,Kota Denpasar,indoor,4.8,1403,0.272188,0.445396,1.12,0.498844,cuaca hujan lebih cocok untuk destinasi dalam ruangan
475,Skenario 2 - Wisata budaya saat hujan,wisata budaya museum pura sejarah indoor,hujan,False,False,Museum Puri Lukisan,Budaya,Gianyar,Kabupaten Gianyar,indoor,4.3,2195,0.289933,0.441698,1.12,0.494702,cuaca hujan lebih cocok untuk destinasi dalam ruangan
317,Skenario 2 - Wisata budaya saat hujan,wisata budaya museum pura sejarah indoor,hujan,False,False,Museum Yadnya,Budaya,Mengwi,Kabupaten Badung,indoor,4.8,20,0.312107,0.436884,1.12,0.489310,cuaca hujan lebih cocok untuk destinasi dalam ruangan
1118,Skenario 2 - Wisata budaya saat hujan,wisata budaya museum pura sejarah indoor,hujan,False,False,Museum Kertagosa,Budaya,Klungkung,Kabupaten Klungkung,indoor,4.7,81,0.299484,0.435865,1.12,0.488169,cuaca hujan lebih cocok untuk destinasi dalam ruangan



Skenario 3 - Wisata keluarga saat weekend dan high season


,nama_skenario,query,weather,is_weekend,is_high_season,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,tipe_wisata,rating,jumlah_rating,cbf_score,base_score,context_multiplier,final_score,reason
119,Skenario 3 - Wisata keluarga saat weekend dan high season,wisata keluarga rekreasi taman edukasi,berawan,True,True,Wisata Edukasi Subak Teba Majalangu,Umum,Denpasar Timur,Kota Denpasar,indoor,4.7,504,0.261437,0.425001,1.0400,0.442001,cuaca berawan tidak memberi perubahan besar; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
1350,Skenario 3 - Wisata keluarga saat weekend dan high season,wisata keluarga rekreasi taman edukasi,berawan,True,True,Agro Wisata Dan Edukasi Lembah Permai,Umum,Jembrana,Kabupaten Jembrana,mixed,5.0,3,0.257753,0.392453,1.0400,0.408151,cuaca berawan tidak memberi perubahan besar; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
631,Skenario 3 - Wisata keluarga saat weekend dan high season,wisata keluarga rekreasi taman edukasi,berawan,True,True,Taman Wisata Luih,Rekreasi,Baturiti,Kabupaten Tabanan,outdoor,4.6,135,0.205203,0.370257,1.0608,0.392769,cuaca berawan masih cukup cocok untuk destinasi luar ruangan; akhir pekan dan destinasi tidak terlalu populer sehing...
109,Skenario 3 - Wisata keluarga saat weekend dan high season,wisata keluarga rekreasi taman edukasi,berawan,True,True,Taman Wisata mangrove,Rekreasi,Denpasar Timur,Kota Denpasar,mixed,4.6,57,0.193593,0.354737,1.0400,0.368927,cuaca berawan tidak memberi perubahan besar; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
134,Skenario 3 - Wisata keluarga saat weekend dan high season,wisata keluarga rekreasi taman edukasi,berawan,True,True,Taman Mumbul Sangeh,Rekreasi,Abiansemal,Kabupaten Badung,outdoor,4.6,2750,0.133114,0.345880,1.0608,0.366909,cuaca berawan masih cukup cocok untuk destinasi luar ruangan; akhir pekan dan destinasi tidak terlalu populer sehing...



Skenario 4 - Wisata pantai saat hujan


,nama_skenario,query,weather,is_weekend,is_high_season,nama_tempat_wisata,kategori,kecamatan,kabupaten_kota,tipe_wisata,rating,jumlah_rating,cbf_score,base_score,context_multiplier,final_score,reason
810,Skenario 4 - Wisata pantai saat hujan,wisata pantai laut outdoor,hujan,True,False,Wisata Pantai penimbangan,Alam,Buleleng,Kabupaten Buleleng,outdoor,4.8,20,0.374144,0.480311,0.7488,0.359657,cuaca hujan kurang cocok untuk destinasi luar ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
1360,Skenario 4 - Wisata pantai saat hujan,wisata pantai laut outdoor,hujan,True,False,Monumen Operasi Lintas Laut Jawa - Bali,Alam,Jembrana,Kabupaten Jembrana,mixed,4.4,219,0.180096,0.348854,0.9568,0.333783,cuaca hujan cukup aman untuk destinasi campuran; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
69,Skenario 4 - Wisata pantai saat hujan,wisata pantai laut outdoor,hujan,True,False,Wisata Bali Penida,Umum,Denpasar Selatan,Kota Denpasar,indoor,4.9,70,0.057776,0.273420,1.1648,0.318480,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
119,Skenario 4 - Wisata pantai saat hujan,wisata pantai laut outdoor,hujan,True,False,Wisata Edukasi Subak Teba Majalangu,Umum,Denpasar Timur,Kota Denpasar,indoor,4.7,504,0.042795,0.271951,1.1648,0.316769,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
5,Skenario 4 - Wisata pantai saat hujan,wisata pantai laut outdoor,hujan,True,False,Lapangan Puputan Badung (I Gusti Ngurah Made Agung),Umum,Denpasar Barat,Kota Denpasar,indoor,4.7,3909,0.000000,0.259749,1.1648,0.302556,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...



Hasil pengujian skenario berhasil disimpan ke:
../data/hasil_test_skenario_cbf_cars.csv


## 24. Perbandingan CBF Saja vs CBF + CARS

Bagian ini membuktikan bahwa CARS dapat mengubah urutan rekomendasi berdasarkan konteks.


In [25]:
comparison_query = "wisata pantai alam outdoor"
comparison_weather = "hujan"

cbf_only = recommend_cbf(comparison_query, top_n=10).copy()
cbf_cars = recommend_cbf_cars(
    comparison_query,
    weather=comparison_weather,
    is_weekend=True,
    is_high_season=False,
    top_n=10
).copy()

cbf_only_display = cbf_only[["nama_tempat_wisata", "tipe_wisata", "cbf_score", "base_score"]].copy()
cbf_only_display.insert(0, "ranking_cbf", range(1, len(cbf_only_display) + 1))

cbf_cars_display_cols = ["nama_tempat_wisata", "tipe_wisata", "base_score", "context_multiplier", "final_score", "reason"]
cbf_cars_display_cols = [col for col in cbf_cars_display_cols if col in cbf_cars.columns]
cbf_cars_display = cbf_cars[cbf_cars_display_cols].copy()
cbf_cars_display.insert(0, "ranking_cbf_cars", range(1, len(cbf_cars_display) + 1))

print("CBF saja:")
display(cbf_only_display)

print("CBF + CARS dengan konteks cuaca hujan:")
display(cbf_cars_display)


CBF saja:


,ranking_cbf,nama_tempat_wisata,tipe_wisata,cbf_score,base_score
810,1,Wisata Pantai penimbangan,outdoor,0.386402,0.488891
55,2,Pantai,mixed,0.388733,0.358126
201,3,Pantai Pandawa,outdoor,0.092867,0.341912
250,4,Pantai Kuta,outdoor,0.095656,0.339382
254,5,Pantai Melasti Ungasan,outdoor,0.076039,0.332809
206,6,Pantai Kuta Bali,outdoor,0.087703,0.327591
340,7,Desa Wisata Penglipuran,mixed,0.061887,0.324716
219,8,Pantai Mengening,outdoor,0.089984,0.317035
225,9,Pantai Petitenget,outdoor,0.088974,0.315227
303,10,Pantai Batu Bolong,outdoor,0.075582,0.310797


CBF + CARS dengan konteks cuaca hujan:


,ranking_cbf_cars,nama_tempat_wisata,tipe_wisata,base_score,context_multiplier,final_score,reason
810,1,Wisata Pantai penimbangan,outdoor,0.488891,0.7488,0.366082,cuaca hujan kurang cocok untuk destinasi luar ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
55,2,Pantai,mixed,0.358126,0.9568,0.342655,cuaca hujan cukup aman untuk destinasi campuran; akhir pekan dan destinasi tidak terlalu populer sehingga lebih nyaman
69,3,Wisata Bali Penida,indoor,0.272864,1.1648,0.317832,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
119,4,Wisata Edukasi Subak Teba Majalangu,indoor,0.271540,1.1648,0.316289,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
5,5,Lapangan Puputan Badung (I Gusti Ngurah Made Agung),indoor,0.259749,1.1648,0.302556,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
79,6,Vihara Satya Dharma,indoor,0.259244,1.1648,0.301967,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
15,7,Lapangan Puputan Renon,indoor,0.256293,1.1648,0.298530,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
118,8,Museum Bung Karno,indoor,0.254865,1.1648,0.296866,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
48,9,Dewata Bali Travel - Paket Wisata dan Liburan di Bali,indoor,0.253920,1.1648,0.295766,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...
1126,10,Pasar Umum Klungkung,indoor,0.249800,1.1648,0.290967,cuaca hujan lebih cocok untuk destinasi dalam ruangan; akhir pekan dan destinasi tidak terlalu populer sehingga lebi...


## 25. Kesimpulan Notebook

Berdasarkan proses yang dilakukan, notebook ini dapat digunakan sebagai bukti bahwa:

1. Dataset `bali_tourist_destination.csv` telah dibersihkan.
2. Missing value, duplikat, rating, jumlah rating, dan koordinat telah diperiksa.
3. Dataset bersih telah disimpan sebagai `cleaned_dataaset_bali.csv`.
4. Dataset dapat digunakan untuk proses CBF dengan TF-IDF dan cosine similarity.
5. Hasil CBF dapat disesuaikan menggunakan CARS berdasarkan cuaca, weekend, high season, tipe wisata, dan popularitas.
6. CARS tidak digunakan untuk prediksi cuaca, melainkan sebagai penyesuaian konteks rekomendasi.

Kalimat aman untuk skripsi:

> Dataset destinasi wisata Bali terlebih dahulu melalui tahap preprocessing yang meliputi pemeriksaan missing value, penghapusan data duplikat, validasi rating, validasi jumlah ulasan, validasi koordinat lokasi, pembuatan tipe wisata, serta pembentukan feature_text. Feature_text digunakan pada proses Content-Based Filtering menggunakan TF-IDF dan cosine similarity. Selanjutnya, hasil rekomendasi disesuaikan menggunakan Context-Aware Recommender System berdasarkan konteks cuaca, weekend, high season, tipe wisata, dan popularitas destinasi.
